# Dynamic Treasury Hedging of Agency MBS
### 00. Project guide

**Olti Myrtaj** · Erdős Institute Quantitative Finance Boot Camp, Fall 2026

> **When does refinancing information help a mortgage investor adjust a Treasury hedge beyond what a simple updating hedge already achieves, and how much improvement survives realistic trading costs?**

This notebook is the map. It explains the financial problem in plain language, defines every term the other notebooks use, shows the frozen experiment settings, and checks which parts of the project have been run.

## 1. The problem in one picture

A Treasury bond pays fixed cash flows. Its interest-rate exposure (duration) changes slowly and predictably as it ages.

An Agency mortgage-backed security (MBS) passes homeowners' mortgage payments to investors. Every homeowner holds an option: they can repay the loan early, usually by refinancing when mortgage rates fall. That option changes the bond's exposure:

* **Rates fall** → refinancing becomes attractive → principal comes back early → the MBS behaves like a short bond → **duration falls**, just when a long position would have gained the most.
* **Rates rise** → nobody refinances ("locked in") → principal comes back slowly → **duration rises**, just when losses are largest.

So a manager who sized a Treasury hedge to last month's exposure can find it too large after a rally and too small after a selloff. This shape is called **negative convexity**, and it is why mortgage desks re-hedge.

The project asks a practical follow-up: a simple trailing estimate of exposure already updates the hedge. Does knowing *how many borrowers can profitably refinance* make the hedge better than that, after paying to trade?

## 2. The four questions and where they are answered

| # | Question | Evidence | Notebook |
|---|---|---|---|
| Q1 | Does measured MBS rate exposure change meaningfully through time? | Monthly empirical duration with error bars, and a test for variation beyond noise | 05 (development), 08 (full sample) |
| Q2 | Does refinancing information forecast next month's exposure beyond persistence or rolling estimation? | Walk-forward forecast errors and the Clark–West test | 06, 08 |
| Q3 | Does it lower out-of-sample hedged risk against static and updating hedges? | One backtest engine, bootstrap intervals | 07, 08 |
| Q4 | Where does any improvement justify its turnover and cost? | Cost grid, rebalancing rules, regimes, and episodes | 08 |

Supporting notebooks: 01 (data), 02 (panels and timing), 03 (teaching example), 04 (simulation checks), 09 (VMBS and PIMCO), 10 (final figures, claim ledger, and return bundle).

## 3. How this project meets the Fall 2026 requirements

| Requirement (Fall 2026 Project Instructions) | Where it is met |
|---|---|
| A clearly stated financial or quantitative question | Section 1 above and `docs/research_design.md` |
| An appropriate mathematical, statistical, or computational method | Rolling and monthly regressions, walk-forward forecasting, a DV01-matched hedge, block bootstrap (notebooks 05 to 08) |
| Real or realistically simulated data | FRED, Freddie Mac PMMS, FHFA NMDB, and ETF prices (01, 02). Synthetic data appears only in the smoke run and in notebook 04, always labeled. |
| Validation, benchmarking, or testing | Static, updating, persistence, and AR(1) benchmarks, known-truth simulation, look-ahead perturbation tests, and `pytest` |
| Financial interpretation | Every result is reported in risk, turnover, and cost units a portfolio manager uses ($ per $100mm) |
| Assumptions, limitations, and failure modes | `docs/assumptions_limitations.md`, linked to evidence in notebook 10 |
| Reproducible annotated notebooks, README, executive summary, and video | This sequence, `README.md`, `RUN_ME_FIRST.md`. The summary and video come in the next stage from the saved evidence. |

## 4. Glossary

| Term | Meaning in this project | Units |
|---|---|---|
| Excess return | Return minus the 3-month Treasury bill accrual over the same days | decimal |
| Empirical duration | How much the MBS excess return falls per 1-unit rise in the 10-year yield, estimated by regression on daily data. It includes prepayment effects and spread moves that co-move with rates. | years |
| Effective (option-adjusted) duration | A model-based duration from a prepayment and rate model with spreads held fixed (for example the 5.71 years iShares reported for MBB on Oct 29, 2026). Not estimated here. | years |
| DV01 | Dollar change in value for a 1 bp move in yield = duration × value × 0.0001 | $ per bp |
| Hedge ratio h | Dollars of 10-year Treasury sold short per $1 of MBS, chosen so the DV01s match: h = forecast MBS duration ÷ hedge-bond duration | $ per $ |
| Refinancing incentive I | Average note rate on outstanding agency-eligible mortgages minus the current 30-year PMMS rate | percentage points |
| Share in the money g | Estimated share of borrowers whose note rate is at least θ = 0.5 pp above today's rate, g = Φ((I − θ)/σ) | 0 to 1 |
| Static hedge (S) | Hedge sized to the average development-period duration and never re-estimated | |
| Updating hedge (U) | Hedge sized to an exponentially weighted regression duration, refreshed monthly | |
| Economic state hedge (E) | Hedge sized from the refinancing state alone | |
| Hybrid hedge (H) | Updating estimate plus the duration change the refinancing state implies since that estimate was formed. The headline challenger. | |
| Turnover | Sum of absolute changes in hedge notional per year, as a multiple of MBS notional | × notional per year |
| Cost drag | Trading cost per year from turnover at the assumed one-way cost | bp per year |
| Variance reduction | 1 − Var(challenger) ÷ Var(comparator), on weekly hedged excess returns | fraction |
| Walk-forward | Every forecast uses only data available at its decision date, with models refit each month on an expanding window | |
| Freeze | The record, written before any test-period result exists, of every development choice and the hashes of the configuration, the result-relevant code, and the analysis notebooks' code | |

In [ ]:
# Setup: find the project root, load the configuration, and pick the run mode.
# The mode comes from MBSHEDGE_MODE (set by run_project.py) or config/project.yaml.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config" / "project.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mbshedge.config import setup, experiment
from mbshedge import plotting as pt

cfg, paths, MODE = setup()
EXP = experiment(cfg, MODE)
pt.setup_style()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

## 5. The frozen experiment settings

These settings come straight from `config/project.yaml`. Notebook 06 hashes this block into the freeze record, so after the freeze any change is detected and must be logged.

In [ ]:
e = cfg["experiment"]
settings = pd.DataFrame([
    ("Hedge instrument", f"synthetic {e['hedge_tenor_years']}-year constant-maturity par Treasury (robustness: {e['hedge_tenor_robustness']}-year)"),
    ("Rate factor", e["rate_factor_series"]),
    ("Notional for dollar reporting", f"${e['notional_usd']:,.0f}"),
    ("Measurement start", e["sample"]["first_month"]),
    ("Development period ends", e["sample"]["development_end"]),
    ("Final test starts", e["sample"]["test_start"]),
    ("Walk-forward burn-in (months)", e["sample"]["min_training_months"]),
    ("EWMA half-life grid (trading days)", e["exposure"]["ewma_halflife_grid_days"]),
    ("Refinancing threshold theta (pp)", e["incentive"]["refi_threshold_pp"]),
    ("Models", ", ".join(e["models"]["list"])),
    ("Headline challenger", e["models"]["headline_challenger"]),
    ("Primary comparisons", "; ".join(f"{a} vs {b}" for a, b in e["models"]["primary_comparisons"])),
    ("Base rebalancing / delay / cost", f"{e['hedging']['base_rule']} / {e['hedging']['base_delay_days']} day / {e['hedging']['base_cost_bp']} bp"),
    ("Cost grid (bp per unit traded)", e["hedging"]["cost_grid_bp"]),
    ("Rebalancing rules", e["hedging"]["rebalance_rules"]),
    ("Bootstrap", f"{e['evaluation']['bootstrap_reps']} reps, {e['evaluation']['bootstrap_block_weeks']}-week mean block"),
    ("Hurdle information ratio (cost verdict)", e["evaluation"]["hurdle_information_ratio"]),
    ("Practical targets", f"{e['evaluation']['practical_variance_reduction_vs_U']:.0%} vs U, {e['evaluation']['practical_variance_reduction_vs_S']:.0%} vs S"),
    ("NMDB availability lag (quarters)", cfg["timing"]["nmdb_lag_quarters"]),
    ("Seed", e["seed"]),
], columns=["Setting", "Value"])
settings

In [ ]:
print("Episodes declared before any test-period evaluation:")
pd.DataFrame(cfg["experiment"]["episodes"])

## 6. How to run the project

1. Create the environment and install the packages (see `RUN_ME_FIRST.md`).
2. From the project folder run `python run_project.py --mode smoke` once. It uses synthetic data, takes about two minutes, and proves the installation works. Every output it writes is under `outputs/smoke/` and is watermarked SYNTHETIC.
3. Run `python run_project.py --mode full`. It downloads the data, builds the panels, runs the simulations, freezes the experiment in notebook 06, evaluates the test period, and writes the return bundle. Expected time: about 10 minutes, most of it downloads and simulations.
4. Optional: `--mode development` stops at the freeze, so you can read the development results before the test period is opened.

You can also open the notebooks in Jupyter and run them in order 00 → 10. Each notebook loads its inputs from files written by earlier notebooks, so nothing depends on hidden kernel state.

## 7. Run status

In [ ]:
from mbshedge import freeze as fz
checks = {
    "01 raw manifest": paths.raw / "manifest.csv",
    "01 access status": paths.results / "data_status" / "access_status.csv",
    "02 daily panel": paths.processed / "daily_panel.parquet",
    "04 simulation summary": paths.results / "simulation" / "power_summary.json",
    "05 monthly exposure panel": paths.processed / "monthly_panel.parquet",
    "06 freeze record": fz.record_path(paths),
    "07 test backtest": paths.results / "backtest" / "weekly_test_base.parquet",
    "08 main table": paths.results / "evaluation" / "main_table_test.csv",
    "09 external validation": paths.results / "external" / "external_status.json",
    "10 claim ledger": paths.results / "claims" / "claim_ledger.csv",
}
status = pd.DataFrame([(k, "done" if v.exists() else "not yet", paths.rel(v)) for k, v in checks.items()],
                      columns=["Step", "Status", "File"])
status

## 8. Records to read alongside the notebooks

* `docs/research_design.md`: the professional framing, methods, and success definitions
* `docs/decision_log.md`: every change from the original design PDF, with reasons, and the mapping of its hypotheses
* `docs/data_sources.md` and `docs/data_dictionary.md`: what was verified, how to supply files by hand, and every field's units and availability rule
* `docs/assumptions_limitations.md`: what could make the conclusions wrong
* `docs/experiment_freeze.md`: how the freeze works
* `docs/reference_review.md`: what the strongest past projects did well
* `docs/presentation_template_notes.md`: the 2025 deck and summary templates for the next stage